# Macro-Aware Credit Risk: Does the Local / Systematic Economy at Origination Improve Default Prediction?
 
**Course Alignment:** ES 335 Machine Learning | IIT Gandhinagar  
**Core Research Question:** Does incorporating macroeconomic conditions available at loan origination significantly improve retail loan default prediction (in ranking discrimination via PR-AUC/ROC-AUC and probability calibration via Brier score) beyond individual borrower and loan application characteristics?

---
### End-to-End Quantitative Pipeline Overview (Audited & Methodologically Frozen)
1. **Phase 1: Environment Setup & Toolchain Initialization** (Python 3.12, scikit-learn, statsmodels, LightGBM, plotting styles)
2. **Phase 2: Data Procurement** (LendingClub 2.26M loans micro-data & FRED 144-month macroeconomic series)
3. **Phase 3: Data Cleaning & Point-in-Time (PIT) Chronological Merging** (Matured 36-month cohort, Dec 2015 maturity cutoff, 2-month publication lag)
4. **Phase 4: Feature Engineering & Preprocessing** (Separated feature groups, fold-specific pipelines, zero leakage, optional ablations)
5. **Phase 5: Model Training & Out-of-Time (OOT) Validation** (Expanding-window CV on 2007–2014, unweighted calibration preservation, Models A–D)
6. **Phase 6: Evaluation Metrics, Calibration & Statistical Uncertainty** (Untouched 2015 holdout test, PR-AUC, Brier score, month-clustered SEs & block bootstrap)
7. **Phase 7: Origination-Time Sensitivity Explorer & Web Model Export** (Export scaler parameters and coefficients to `web/`, client-side JS verification)

## Phase 1: Environment Setup & Toolchain Initialization

In [19]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, 
    precision_recall_curve, 
    auc, 
    brier_score_loss, 
    classification_report, 
    confusion_matrix
)
from sklearn.calibration import calibration_curve

# Set plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 50)
print('All primary libraries imported successfully.')

All primary libraries imported successfully.


## Phase 2: Data Procurement (Raw Materials)
- **Borrower Micro-Data:** LendingClub Loan Dataset (origination dates `issue_d`, FICO, DTI, Income, target `loan_status`)
- **Macroeconomic Series:** FRED API (`UNRATE`, `CPIAUCSL`, `FEDFUNDS`)

In [20]:
# Phase 2: Load Curated Datasets
# 1. Micro-Data (LendingClub filtered to 15 relevant features)
microdata_path = 'data/processed/microdataprocessed.csv'
print(f'Loading micro-data from {microdata_path}...')
microdata = pd.read_csv(microdata_path, low_memory=False)
print(f'Micro-data loaded: {microdata.shape[0]:,} rows, {microdata.shape[1]} columns.')

# 2. Macroeconomic Series (FRED monthly indicators 2007-2018)
macrodata_path = 'data/processed/macrodata.csv'
print(f'Loading macro-data from {macrodata_path}...')
macrodata = pd.read_csv(macrodata_path)
print(f'Macro-data loaded: {macrodata.shape[0]} months, {macrodata.shape[1]} columns.')


Loading micro-data from data/processed/microdataprocessed.csv...
Micro-data loaded: 2,260,701 rows, 15 columns.
Loading macro-data from data/processed/macrodata.csv...
Macro-data loaded: 144 months, 7 columns.


In [21]:
# Display previews of loaded datasets
print('=== LendingClub Micro-Data Preview ===')
display(microdata.head())

print('=== FRED Macroeconomic Data Preview ===')
display(macrodata.head())


=== LendingClub Micro-Data Preview ===


,loan_amnt,term,int_rate,installment,home_ownership,annual_inc,issue_d,loan_status,dti,fico_range_low,fico_range_high,open_acc,pub_rec,revol_util,mort_acc
0,3600.0,36 months,13.99,123.03,MORTGAGE,55000.0,Dec-2015,Fully Paid,5.91,675.0,679.0,7.0,0.0,29.7,1.0
1,24700.0,36 months,11.99,820.28,MORTGAGE,65000.0,Dec-2015,Fully Paid,16.06,715.0,719.0,22.0,0.0,19.2,4.0
2,20000.0,60 months,10.78,432.66,MORTGAGE,63000.0,Dec-2015,Fully Paid,10.78,695.0,699.0,6.0,0.0,56.2,5.0
3,35000.0,60 months,14.85,829.90,MORTGAGE,110000.0,Dec-2015,Current,17.06,785.0,789.0,13.0,0.0,11.6,1.0
4,10400.0,60 months,22.45,289.91,MORTGAGE,104433.0,Dec-2015,Fully Paid,25.37,695.0,699.0,12.0,0.0,64.5,6.0


=== FRED Macroeconomic Data Preview ===


,date,UNRATE,CPIAUCSL,FEDFUNDS,UMCSENT,DGS10,CPI_YoY_Inflation
0,2007-01-01,4.6,203.437,5.25,96.9,4.76,2.08
1,2007-02-01,4.5,204.226,5.26,91.3,4.72,2.42
2,2007-03-01,4.4,205.288,5.26,88.4,4.56,2.80
3,2007-04-01,4.5,205.904,5.25,87.1,4.69,2.59
4,2007-05-01,4.4,206.755,5.25,88.3,4.75,2.71


## Phase 3: Data Cleaning & Point-in-Time (PIT) Chronological Merging
Key integrity requirements:
1. **Target Filtering:** Retain only completed outcomes: `Charged Off` (1) and `Fully Paid` (0). Exclude ongoing `Current` loans to prevent right-censoring bias.
2. **Eliminate Future Leakage:** Drop post-origination columns (`recoveries`, `total_pymnt`, `last_pymnt_d`, etc.) — already secured via the 15-feature application-time allowlist.
3. **Publication Lag:** Shift macroeconomic indicators by 1 month (`macro_df[indicators].shift(1)`) to reflect realistic economic reporting lags.
4. **Point-in-Time Merge:** Join borrower micro-data with prevailing lagged macro indicators based on loan origination month (`issue_d`).

In [22]:
# 1. Clean missing origination dates, filter completed outcomes, and convert issue_d to datetime
print('Preparing micro-data dates and filtering terminal outcomes...')
micro_df = microdata.dropna(subset=['issue_d']).copy()

# Filter completed loan outcomes to prevent target censoring (Fully Paid vs Charged Off)
terminal_statuses = ['Fully Paid', 'Charged Off']
micro_df = micro_df[micro_df['loan_status'].isin(terminal_statuses)].copy()

# Encode binary target: 1 for Charged Off (Default), 0 for Fully Paid
micro_df['default'] = (micro_df['loan_status'] == 'Charged Off').astype(int)

# Standardize origination date to monthly datetime (start of month)
micro_df['issue_date'] = pd.to_datetime(micro_df['issue_d'], format='%b-%Y')
micro_df = micro_df.sort_values('issue_date').reset_index(drop=True)

# 2. Prepare macro-data with 1-month publication lag
print('Applying 1-month publication lag to macro indicators...')
macro_df = macrodata.copy()
macro_df['macro_date'] = pd.to_datetime(macro_df['date'])
macro_df = macro_df.sort_values('macro_date').reset_index(drop=True)

macro_indicators = ['UNRATE', 'CPIAUCSL', 'FEDFUNDS', 'UMCSENT', 'DGS10', 'CPI_YoY_Inflation']
macro_df[macro_indicators] = macro_df[macro_indicators].shift(1)

# 3. Point-in-Time (PIT) Merge: Match each loan to prevailing macro climate
print('Merging micro-data and macro-data into unified dataframe df...')
df = pd.merge(
    micro_df,
    macro_df.drop(columns=['date']),
    left_on='issue_date',
    right_on='macro_date',
    how='inner'
).drop(columns=['macro_date'])

# Drop any edge records with missing macro data resulting from the 1-month lag
df = df.dropna(subset=macro_indicators).reset_index(drop=True)

print(f'Successfully created unified dataframe df: {df.shape[0]:,} rows, {df.shape[1]} columns.')


Preparing micro-data dates and filtering terminal outcomes...
Applying 1-month publication lag to macro indicators...
Merging micro-data and macro-data into unified dataframe df...
Successfully created unified dataframe df: 1,345,310 rows, 23 columns.


In [23]:
# Display preview and summary diagnostics of unified dataset
print('=== Unified Micro + Macro Dataset Preview (First 5 Records) ===')
display(df.head())

print('\n=== Dataset Diagnostics & Integrity Checks ===')
print(f'Total observations: {len(df):,}')
print(f'Origination date span: {df["issue_date"].min().strftime("%b %Y")} to {df["issue_date"].max().strftime("%b %Y")}')
print(f'Target distribution: {df["default"].value_counts().to_dict()} (Default Rate: {df["default"].mean():.2%})')
print(f'Missing values across macro indicators: {df[macro_indicators].isna().sum().to_dict()}')


=== Unified Micro + Macro Dataset Preview (First 5 Records) ===


,loan_amnt,term,int_rate,installment,home_ownership,annual_inc,issue_d,loan_status,dti,fico_range_low,fico_range_high,open_acc,pub_rec,revol_util,mort_acc,default,issue_date,UNRATE,CPIAUCSL,FEDFUNDS,UMCSENT,DGS10,CPI_YoY_Inflation
0,7500.0,36 months,13.75,255.43,OWN,22000.0,Jun-2007,Fully Paid,14.29,660.0,664.0,7.0,0.0,51.5,NaN,0,2007-06-01,4.4,206.755,5.25,88.3,4.75,2.71
1,3500.0,36 months,10.28,113.39,RENT,20000.0,Jul-2007,Fully Paid,1.50,680.0,684.0,17.0,0.0,32.4,NaN,0,2007-07-01,4.6,207.234,5.25,85.3,5.10,2.69
2,5750.0,36 months,7.43,178.69,MORTGAGE,125000.0,Jul-2007,Fully Paid,0.27,790.0,794.0,10.0,0.0,10.2,NaN,0,2007-07-01,4.6,207.234,5.25,85.3,5.10,2.69
3,5000.0,36 months,7.43,155.38,RENT,40000.0,Jul-2007,Fully Paid,2.55,770.0,774.0,4.0,0.0,14.0,NaN,0,2007-07-01,4.6,207.234,5.25,85.3,5.10,2.69
4,1200.0,36 months,11.54,39.60,RENT,20000.0,Jul-2007,Fully Paid,2.04,660.0,664.0,3.0,0.0,75.8,NaN,0,2007-07-01,4.6,207.234,5.25,85.3,5.10,2.69



=== Dataset Diagnostics & Integrity Checks ===
Total observations: 1,345,310
Origination date span: Jun 2007 to Dec 2018
Target distribution: {0: 1076751, 1: 268559} (Default Rate: 19.96%)
Missing values across macro indicators: {'UNRATE': 0, 'CPIAUCSL': 0, 'FEDFUNDS': 0, 'UMCSENT': 0, 'DGS10': 0, 'CPI_YoY_Inflation': 0}


## Phase 4: Cohort Isolation, Feature Engineering & Preprocessing

### Methodological Principles & Preprocessing Architecture
1. **Censoring-Free 36-Month Matured Cohort Isolation (Change 1):**
   - Restricts the dataset strictly to **36-month installment loans** (`term` contains `'36'`) originated on or before **`2015-12-31`** (`MATURITY_CUTOFF_DATE`), where unresolved loans drop below 0.5%.
   - Maps verified terminal outcomes (`Fully Paid`, `Charged Off`, `Default`, and 2007–2011 legacy credit policy outcomes) into binary target `default` $\in \{0, 1\}$ and deduplicates exact duplicates.
   - Aligns macroeconomic features (`UNRATE`, `UNRATE_change_6m`, `CPI_YoY_Inflation`) using a conservative **2-month publication lag (`shift(2)`)** via `src/macro_processing.py`.
2. **Strict Feature Group Separation:**
   - **Group A (Borrower-Only Pre-Origination Allowlist):** `annual_inc`, `dti`, `fico_range_low`, `fico_range_high`, `open_acc`, `pub_rec`, `revol_util`, `mort_acc`, `loan_amnt`, `home_ownership`.
   - **Group B (Macroeconomic Series):** 2-month lagged stationary indicators: `UNRATE`, `UNRATE_change_6m`, `CPI_YoY_Inflation`.
   - **Group C (Lender-Information Benchmark):** `int_rate`, `installment`. *(Explicitly excluded from borrower-only baseline; evaluated separately)*.
3. **Interaction Policy & Ablation Experiments:**
   - Primary Models (`MODEL_C` and `MODEL_D`) use strictly un-interacted additive features to maintain clean interpretability.
   - Two economically motivated hypotheses are engineered for **optional ablation experiments**:
     * `dti_x_unemp = dti * UNRATE`: Tests if debt-burdened borrowers exhibit higher default sensitivity during labor-market downturns.
     * `fico_x_unemp = fico_avg * UNRATE`: Tests if prime credit cushions provide asymmetric protection against systematic economic shocks.
4. **Fold-Specific Preprocessing (Zero Data Leakage) & Integrity Audit:**
   - All preprocessing transformers are constructed using `sklearn.pipeline.Pipeline` inside a `ColumnTransformer`.
   - Preprocessing is fitted **strictly on the training partition of each expanding-window fold**.
   - Verified against all 6 automated checks in `src/validation_checks.py`.

In [24]:
# Phase 4: Matured Cohort Isolation, Feature Engineering & Pipeline Construction
# Note: All preprocessing transformers remain unfitted here and will be fit strictly inside CV training folds.

import sys
import os

# Invalidate any stale namespace module cached in an active Jupyter kernel
if 'config' in sys.modules and not hasattr(sys.modules['config'], 'BORROWER_NUMERICAL_FEATURES'):
    del sys.modules['config']

# Ensure project root and src/ directories are in Python search path
for p in ['.', 'src']:
    abs_p = os.path.abspath(p)
    if abs_p not in sys.path:
        sys.path.insert(0, abs_p)

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from config import (
    BORROWER_NUMERICAL_FEATURES,
    BORROWER_CATEGORICAL_FEATURES,
    PRIMARY_MACRO_FEATURES,
    FEATURE_GROUPS,
    OPTIONAL_ABLATIONS,
    MATURITY_CUTOFF_DATE,
    RESOLVED_DEFAULT_STATUSES,
    RESOLVED_PAID_STATUSES,
    TRAIN_START_DATE,
    TRAIN_END_DATE,
    TEST_START_DATE,
    TEST_END_DATE
)
from macro_processing import prepare_macro_data, merge_micro_and_macro
from validation_checks import run_all_validation_checks

# ------------------------------------------------------------------------------
# 1. Change 1: Censoring-Free 36-Month Matured Cohort & 2-Month Lag Alignment
# ------------------------------------------------------------------------------
print('Applying Change 1: Filtering to matured 36-month cohort (<= 2015-12-31)...')
cohort = microdata.dropna(subset=['issue_d']).copy()
cohort = cohort[cohort['term'].str.contains('36', na=False)].copy()
cohort['issue_date'] = pd.to_datetime(cohort['issue_d'], format='%b-%Y')
cohort = cohort[cohort['issue_date'] <= MATURITY_CUTOFF_DATE].copy()

# Retain all verified terminal outcomes (Charged Off, Default, Fully Paid & legacy credit-policy statuses)
resolved_statuses = RESOLVED_DEFAULT_STATUSES + RESOLVED_PAID_STATUSES
cohort = cohort[cohort['loan_status'].isin(resolved_statuses)].copy()
cohort = cohort.drop_duplicates().reset_index(drop=True)
cohort['default'] = cohort['loan_status'].isin(RESOLVED_DEFAULT_STATUSES).astype(int)

# Apply stationary macro transformations (CPI_YoY_Inflation, UNRATE_change_6m) & 2-month lag
macro_lagged = prepare_macro_data('data/processed/macrodata.csv')
df = merge_micro_and_macro(cohort, macro_lagged)

print(f'Matured analytical cohort `df` ready: {df.shape[0]:,} loans x {df.shape[1]} columns.')
print(f'Cohort date range: {df["issue_date"].min().strftime("%Y-%m-%d")} to {df["issue_date"].max().strftime("%Y-%m-%d")}')
print(f'Overall cohort default rate: {df["default"].mean():.2%}')

# ------------------------------------------------------------------------------
# 2. Pre-Origination Feature Engineering & Optional Ablation Interactions
# ------------------------------------------------------------------------------
df['fico_avg'] = (df['fico_range_low'] + df['fico_range_high']) / 2.0

for abl_name, (col1, col2) in OPTIONAL_ABLATIONS.items():
    df[abl_name] = df[col1] * df[col2]

print(f'Engineered pre-origination features: fico_avg, {list(OPTIONAL_ABLATIONS.keys())}')

# ------------------------------------------------------------------------------
# 3. Leakage-Free Preprocessing Pipeline Construction
# ------------------------------------------------------------------------------
def build_preprocessing_pipeline(num_features, cat_features, scale_numeric=True):
    """Constructs a leakage-free Scikit-Learn ColumnTransformer pipeline."""
    num_steps = [('imputer', SimpleImputer(strategy='median'))]
    if scale_numeric:
        num_steps.append(('scaler', StandardScaler()))

    num_transformer = Pipeline(steps=num_steps)
    cat_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', num_transformer, num_features),
            ('cat', cat_transformer, cat_features)
        ],
        remainder='drop'
    )
    return preprocessor

# Instantiate unfitted preprocessors for all experimental model specifications
preprocessors = {
    'MODEL_B': build_preprocessing_pipeline(
        FEATURE_GROUPS['MODEL_B']['numerical'],
        FEATURE_GROUPS['MODEL_B']['categorical'],
        scale_numeric=True
    ),
    'MODEL_C': build_preprocessing_pipeline(
        FEATURE_GROUPS['MODEL_C']['numerical'] + FEATURE_GROUPS['MODEL_C']['macro'],
        FEATURE_GROUPS['MODEL_C']['categorical'],
        scale_numeric=True
    ),
    'MODEL_D': build_preprocessing_pipeline(
        FEATURE_GROUPS['MODEL_D']['numerical'] + FEATURE_GROUPS['MODEL_D']['macro'],
        FEATURE_GROUPS['MODEL_D']['categorical'],
        scale_numeric=False  # Tree-based LightGBM operates on unscaled raw values
    ),
    'MODEL_D_BASE': build_preprocessing_pipeline(
        FEATURE_GROUPS['MODEL_B']['numerical'],
        FEATURE_GROUPS['MODEL_B']['categorical'],
        scale_numeric=False
    ),
    'LENDER_BENCHMARK': build_preprocessing_pipeline(
        FEATURE_GROUPS['LENDER_BENCHMARK']['numerical'],
        FEATURE_GROUPS['LENDER_BENCHMARK']['categorical'],
        scale_numeric=True
    ),
    'MODEL_C_ABLATION': build_preprocessing_pipeline(
        FEATURE_GROUPS['MODEL_C']['numerical'] + FEATURE_GROUPS['MODEL_C']['macro'] + list(OPTIONAL_ABLATIONS.keys()),
        FEATURE_GROUPS['MODEL_C']['categorical'],
        scale_numeric=True
    )
}

print('\nConfigured Preprocessors:')
for model_key in preprocessors:
    desc = FEATURE_GROUPS.get(model_key, {}).get('name', model_key)
    print(f' - {model_key}: {desc}')

# ------------------------------------------------------------------------------
# 4. Chronological Split Verification & Preprocessing Dry-Run
# ------------------------------------------------------------------------------
train_df = df[(df['issue_date'] >= TRAIN_START_DATE) & (df['issue_date'] <= TRAIN_END_DATE)].copy()
test_df = df[(df['issue_date'] >= TEST_START_DATE) & (df['issue_date'] <= TEST_END_DATE)].copy()

print(f'\nChronological Split Summary:')
print(f' - Train Set ({TRAIN_START_DATE} to {TRAIN_END_DATE}): {len(train_df):,} loans (Default Rate: {train_df["default"].mean():.2%})')
print(f' - Holdout Test ({TEST_START_DATE} to {TEST_END_DATE}): {len(test_df):,} loans (Default Rate: {test_df["default"].mean():.2%})')

# Dry-run fit on a clone of MODEL_C strictly on training data (test_df remains untouched)
_dry_run_pipe = clone(preprocessors['MODEL_C'])
_X_train_check = _dry_run_pipe.fit_transform(train_df)
_X_test_check = _dry_run_pipe.transform(test_df)
print(f'Dry-run MODEL_C transformed shapes -> Train: {_X_train_check.shape}, Test: {_X_test_check.shape}')
print(f'Transformed feature names ({len(_dry_run_pipe.get_feature_names_out())}): {list(_dry_run_pipe.get_feature_names_out())}')
assert not np.isnan(_X_train_check).any() and not np.isnan(_X_test_check).any(), 'NaNs detected after preprocessing!'
del _dry_run_pipe, _X_train_check, _X_test_check

# ------------------------------------------------------------------------------
# 5. Automated 6-Point Integrity & Leakage Validation Suite
# ------------------------------------------------------------------------------
print()
audit_passed = run_all_validation_checks(df)
assert audit_passed, 'Phase 4 validation checks failed! Review logs above.'


Applying Change 1: Filtering to matured 36-month cohort (<= 2015-12-31)...
Matured analytical cohort `df` ready: 620,860 loans x 25 columns.
Cohort date range: 2007-09-01 to 2015-12-01
Overall cohort default rate: 13.94%
Engineered pre-origination features: fico_avg, ['dti_x_unemp', 'fico_x_unemp']

Configured Preprocessors:
 - MODEL_B: Borrower-Only Logistic Regression
 - MODEL_C: Borrower + Macro Logistic Regression
 - MODEL_D: Borrower + Macro LightGBM
 - MODEL_D_BASE: MODEL_D_BASE
 - LENDER_BENCHMARK: Lender-Information Benchmark
 - MODEL_C_ABLATION: MODEL_C_ABLATION

Chronological Split Summary:
 - Train Set (2007-01-01 to 2014-12-31): 337,834 loans (Default Rate: 13.15%)
 - Holdout Test (2015-01-01 to 2015-12-31): 283,026 loans (Default Rate: 14.89%)
Dry-run MODEL_C transformed shapes -> Train: (337834, 18), Test: (283026, 18)
Transformed feature names (18): ['num__annual_inc', 'num__dti', 'num__fico_range_low', 'num__fico_range_high', 'num__open_acc', 'num__pub_rec', 'num__revol

## Phase 5: Model Training & Chronological Out-of-Time (OOT) Validation

### Experimental Architecture & Model Suite
The experimental design investigates the marginal predictive and calibration uplift of macroeconomic series across linear and non-linear model families:

1. **Model Sequence:**
   - **Model A (Historical Baseline):** Constant default-rate predictor ($\bar{y}_{\text{train}} = 13.20\%$) representing the uninformed null benchmark.
   - **Model B (Borrower-Only Logistic Regression):** Primary credit scorecard benchmark using application-time borrower characteristics only.
   - **Model C (Borrower + Macro Logistic Regression):** Same borrower characteristics plus 2-month lagged stationary macro features (`UNRATE`, `UNRATE_change_6m`, `CPI_YoY_Inflation`).
   - **Model D (Borrower + Macro LightGBM):** Gradient boosted decision tree evaluating potential non-linear macro stress thresholds.
   - **Model D-Base (Borrower-Only LightGBM):** Non-linear borrower benchmark required to isolate macro feature contribution in tree models.
   - **Lender-Information Benchmark:** Model B features + proprietary pricing features (`int_rate`, `installment`, `grade`).
2. **Econometric vs. Machine Learning Modeling:**
   - `statsmodels.api.Logit`: Used for econometric interpretability, odds ratios ($\text{OR} = e^{\beta_j}$), and month-clustered standard errors.
   - `scikit-learn` `LogisticRegression`: Used for out-of-sample probability prediction and metric evaluation.
3. **Unweighted Probability Calibration Preservation:**
   - Models are trained **unweighted (`class_weight=None`)** as the frozen primary standard. Artificially balancing weights shifts logistic intercepts to predict ~50% default probability, distorting continuous Probability of Default (PD) calibration and inflating Brier score loss. Weighted modeling is preserved strictly as an optional ablation / sensitivity check.
4. **Chronological Expanding-Window Validation (2007–2014 Training Period):**
   - **Fold 1:** Train on 2007–2012 $\rightarrow$ Validate on 2013 (100,422 loans)
   - **Fold 2:** Train on 2007–2013 $\rightarrow$ Validate on 2014 (162,570 loans)
   - **Holdout Test Set (2015):** **Strictly frozen and untouched** during model tuning and hyperparameter selection.

In [25]:
# Phase 5: Chronological Expanding-Window Validation & Econometric Inference
# Executes Model A, B, C, D-Base, D, Lender Benchmark, and Ablation across temporal validation folds.
# Holdout test_df (2015) remains strictly untouched during Phase 5 training and validation.

import sys
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
import lightgbm as lgb

if 'config' in sys.modules and not hasattr(sys.modules['config'], 'TRAIN_START_DATE'):
    del sys.modules['config']

for p in ['.', 'src']:
    abs_p = os.path.abspath(p)
    if abs_p not in sys.path:
        sys.path.insert(0, abs_p)

from config import (
    TRAIN_START_DATE,
    TRAIN_END_DATE,
    CV_SPLITS,
    TEST_START_DATE,
    TEST_END_DATE
)

print(f'Frozen Training Window: {TRAIN_START_DATE} to {TRAIN_END_DATE}')
print(f'Untouched Holdout Test Window: {TEST_START_DATE} to {TEST_END_DATE}')
print(f'Expanding-Window Folds: {len(CV_SPLITS)} chronological validation splits configured.')

# Guardrail assertion: confirm holdout boundary is clean before training
assert train_df['issue_date'].max() < pd.Timestamp(TEST_START_DATE), 'Training data overlaps holdout start date!'


def compute_phase5_metrics(y_true, y_prob):
    """Compute imbalanced risk ranking and calibration metrics."""
    return {
        'PR-AUC': float(average_precision_score(y_true, y_prob)),
        'ROC-AUC': float(roc_auc_score(y_true, y_prob)),
        'Brier_Score': float(brier_score_loss(y_true, y_prob)),
        'Mean_Pred_PD': float(np.mean(y_prob)),
        'Actual_DR': float(np.mean(y_true))
    }


def make_lgbm_model(class_weight=None):
    """Regularized LightGBM classifier tuned via expanding-window CV (2013-2014) to limit macro-level overfitting."""
    return lgb.LGBMClassifier(
        class_weight=class_weight,
        n_estimators=200,
        learning_rate=0.03,
        num_leaves=15,
        max_depth=5,
        min_child_samples=200,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        reg_lambda=10.0,
        deterministic=True,
        force_col_wise=True,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )


# ---------------------------------------------------------------------------
# 1. Chronological Expanding-Window Cross-Validation (Strictly <= 2014-12-31)
# ---------------------------------------------------------------------------
LR_MODEL_KEYS = ['MODEL_B', 'MODEL_C', 'LENDER_BENCHMARK', 'MODEL_C_ABLATION']
LGB_MODEL_KEYS = ['MODEL_D_BASE', 'MODEL_D']

cv_records = []

for split in CV_SPLITS:
    fold_id = split['fold']
    f_train = train_df[train_df['issue_date'] <= split['train_end']].copy()
    f_val = train_df[
        (train_df['issue_date'] >= split['val_start']) &
        (train_df['issue_date'] <= split['val_end'])
    ].copy()

    y_tr = f_train['default'].values
    y_va = f_val['default'].values

    print(
        f"\n[Fold {fold_id}] Train <= {split['train_end']}: {len(f_train):,} loans (DR={y_tr.mean():.2%}) | "
        f"Val {split['val_start']}..{split['val_end']}: {len(f_val):,} loans (DR={y_va.mean():.2%})"
    )

    # Model A: Empirical Historical Baseline (constant training default rate)
    p_a = np.full(len(y_va), float(y_tr.mean()))
    cv_records.append({'Fold': fold_id, 'Model': 'MODEL_A', 'Calibration': 'baseline', **compute_phase5_metrics(y_va, p_a)})

    # Linear Scorecards (Primary Standard: class_weight=None)
    for m_key in LR_MODEL_KEYS:
        prep = clone(preprocessors[m_key])
        X_tr = prep.fit_transform(f_train)
        X_va = prep.transform(f_val)
        clf = LogisticRegression(class_weight=None, max_iter=1000, random_state=42)
        clf.fit(X_tr, y_tr)
        p_va = clf.predict_proba(X_va)[:, 1]
        cv_records.append({'Fold': fold_id, 'Model': m_key, 'Calibration': 'unweighted', **compute_phase5_metrics(y_va, p_va)})

    # Diagnostic Sensitivity Check (Change 3): verify why class_weight='balanced' distorts PD & Brier Score
    prep_bal = clone(preprocessors['MODEL_C'])
    X_tr_bal = prep_bal.fit_transform(f_train)
    X_va_bal = prep_bal.transform(f_val)
    clf_bal = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
    clf_bal.fit(X_tr_bal, y_tr)
    p_va_bal = clf_bal.predict_proba(X_va_bal)[:, 1]
    cv_records.append({
        'Fold': fold_id,
        'Model': 'MODEL_C_BALANCED_CHECK',
        'Calibration': 'balanced_diagnostic',
        **compute_phase5_metrics(y_va, p_va_bal)
    })

    # Gradient Boosted Decision Trees (Primary Standard: class_weight=None)
    for m_key in LGB_MODEL_KEYS:
        prep = clone(preprocessors[m_key])
        X_tr = prep.fit_transform(f_train)
        X_va = prep.transform(f_val)
        gbm = make_lgbm_model(class_weight=None)
        gbm.fit(X_tr, y_tr)
        p_va = gbm.predict_proba(X_va)[:, 1]
        cv_records.append({'Fold': fold_id, 'Model': m_key, 'Calibration': 'unweighted', **compute_phase5_metrics(y_va, p_va)})

cv_results_df = pd.DataFrame(cv_records)
cv_results_df['Calibration_Error'] = cv_results_df['Mean_Pred_PD'] - cv_results_df['Actual_DR']

print('\n=== Phase 5 Expanding-Window Validation Results (2013 & 2014 Folds) ===')
display(cv_results_df.round(5))

cv_mean_df = (
    cv_results_df.groupby(['Model', 'Calibration'], sort=False)[['PR-AUC', 'ROC-AUC', 'Brier_Score', 'Mean_Pred_PD', 'Actual_DR', 'Calibration_Error']]
    .mean()
    .reset_index()
)
print('\n=== Mean CV Performance Across Folds 1-2 ===')
display(cv_mean_df.round(5))


# ---------------------------------------------------------------------------
# 2. Final Training on Full 2007-2014 Training Window (Holdout 2015 Untouched)
# ---------------------------------------------------------------------------
y_train_full = train_df['default'].values
baseline_pd = float(y_train_full.mean())

fitted_preprocessors = {}
fitted_models = {}

for m_key in LR_MODEL_KEYS:
    prep = clone(preprocessors[m_key])
    X_train_m = prep.fit_transform(train_df)
    clf = LogisticRegression(class_weight=None, max_iter=1000, random_state=42)
    clf.fit(X_train_m, y_train_full)
    fitted_preprocessors[m_key] = prep
    fitted_models[m_key] = clf

for m_key in LGB_MODEL_KEYS:
    prep = clone(preprocessors[m_key])
    X_train_m = prep.fit_transform(train_df)
    gbm = make_lgbm_model(class_weight=None)
    gbm.fit(X_train_m, y_train_full)
    fitted_preprocessors[m_key] = prep
    fitted_models[m_key] = gbm

print(f'\nFitted {len(fitted_models)} primary/benchmark models strictly on train_df ({len(train_df):,} loans, baseline PD={baseline_pd:.4%}).')
print(f'Holdout test_df ({len(test_df):,} loans) remains untouched for Phase 6.')


# ---------------------------------------------------------------------------
# 3. Econometric Inference: statsmodels.Logit on Model C with Month-Clustered SEs
# ---------------------------------------------------------------------------
X_train_c_df = pd.DataFrame(
    fitted_preprocessors['MODEL_C'].transform(train_df),
    columns=list(fitted_preprocessors['MODEL_C'].get_feature_names_out()),
    index=train_df.index
)

# Drop exact-collinear fico_range_high (constant +4 offset from fico_range_low), reference category
# home_ownership_RENT, and ultra-rare dummies (<50 obs) to ensure full-rank MLE Hessian.
collinear_or_ref = [
    c for c in X_train_c_df.columns
    if 'fico_range_high' in c or c == 'cat__home_ownership_RENT' or (c.startswith('cat__') and X_train_c_df[c].sum() < 50)
]
X_sm_design = sm.add_constant(X_train_c_df.drop(columns=collinear_or_ref))

logit_result = sm.Logit(y_train_full, X_sm_design).fit(
    cov_type='cluster',
    cov_kwds={'groups': train_df['issue_date']},
    disp=False
)

assert logit_result.mle_retvals.get('converged', False), 'statsmodels.Logit failed to converge!'

conf_int = logit_result.conf_int()
logit_summary_df = pd.DataFrame({
    'coef': logit_result.params,
    'clustered_se': logit_result.bse,
    'z_stat': logit_result.tvalues,
    'p_value': logit_result.pvalues,
    'odds_ratio': np.exp(logit_result.params),
    'ci_low_95': conf_int[0],
    'ci_high_95': conf_int[1]
})

print(
    f"\n=== Econometric Model C (statsmodels.Logit, Month-Clustered SEs, "
    f"{train_df['issue_date'].nunique()} monthly clusters) ==="
)
print(f'Converged: {logit_result.mle_retvals["converged"]} | Dropped collinear/rare columns: {collinear_or_ref}')
display(logit_summary_df.round(5))


Frozen Training Window: 2007-01-01 to 2014-12-31
Untouched Holdout Test Window: 2015-01-01 to 2015-12-31
Expanding-Window Folds: 2 chronological validation splits configured.

[Fold 1] Train <= 2012-12-31: 74,842 loans (DR=13.03%) | Val 2013-01-01..2013-12-31: 100,422 loans (DR=12.33%)

[Fold 2] Train <= 2013-12-31: 175,264 loans (DR=12.62%) | Val 2014-01-01..2014-12-31: 162,570 loans (DR=13.73%)

=== Phase 5 Expanding-Window Validation Results (2013 & 2014 Folds) ===


,Fold,Model,Calibration,PR-AUC,ROC-AUC,Brier_Score,Mean_Pred_PD,Actual_DR,Calibration_Error
0,1,MODEL_A,baseline,0.12326,0.50000,0.10812,0.13026,0.12326,0.00700
1,1,MODEL_B,unweighted,0.16767,0.61677,0.10710,0.14919,0.12326,0.02593
2,1,MODEL_C,unweighted,0.16813,0.61719,0.10732,0.15336,0.12326,0.03010
3,1,LENDER_BENCHMARK,unweighted,0.19259,0.65595,0.10589,0.15353,0.12326,0.03027
4,1,MODEL_C_ABLATION,unweighted,0.16936,0.61867,0.10731,0.15415,0.12326,0.03089
5,1,MODEL_C_BALANCED_CHECK,balanced_diagnostic,0.16703,0.61607,0.27597,0.52886,0.12326,0.40560
6,1,MODEL_D_BASE,unweighted,0.17620,0.62679,0.10644,0.14456,0.12326,0.02130
7,1,MODEL_D,unweighted,0.17566,0.62710,0.10625,0.14164,0.12326,0.01838
8,2,MODEL_A,baseline,0.13726,0.50000,0.11854,0.12625,0.13726,-0.01101
9,2,MODEL_B,unweighted,0.20502,0.63407,0.11540,0.13497,0.13726,-0.00229



=== Mean CV Performance Across Folds 1-2 ===


,Model,Calibration,PR-AUC,ROC-AUC,Brier_Score,Mean_Pred_PD,Actual_DR,Calibration_Error
0,MODEL_A,baseline,0.13026,0.50000,0.11333,0.12826,0.13026,-0.00201
1,MODEL_B,unweighted,0.18634,0.62542,0.11125,0.14208,0.13026,0.01182
2,MODEL_C,unweighted,0.18685,0.62598,0.11138,0.14132,0.13026,0.01106
3,LENDER_BENCHMARK,unweighted,0.21283,0.66335,0.10975,0.13939,0.13026,0.00913
4,MODEL_C_ABLATION,unweighted,0.18754,0.62683,0.11131,0.14312,0.13026,0.01286
5,MODEL_C_BALANCED_CHECK,balanced_diagnostic,0.18601,0.62527,0.25885,0.50793,0.13026,0.37767
6,MODEL_D_BASE,unweighted,0.19198,0.63411,0.11072,0.13902,0.13026,0.00875
7,MODEL_D,unweighted,0.19047,0.63155,0.11091,0.14797,0.13026,0.01771



Fitted 6 primary/benchmark models strictly on train_df (337,834 loans, baseline PD=13.1550%).
Holdout test_df (283,026 loans) remains untouched for Phase 6.

=== Econometric Model C (statsmodels.Logit, Month-Clustered SEs, 88 monthly clusters) ===
Converged: True | Dropped collinear/rare columns: ['num__fico_range_high', 'cat__home_ownership_ANY', 'cat__home_ownership_NONE', 'cat__home_ownership_RENT']


,coef,clustered_se,z_stat,p_value,odds_ratio,ci_low_95,ci_high_95
const,-1.89542,0.01266,-149.69854,0.00000,0.15026,-1.92023,-1.87060
num__annual_inc,-0.24263,0.02482,-9.77390,0.00000,0.78456,-0.29129,-0.19398
num__dti,0.14781,0.01071,13.80691,0.00000,1.15929,0.12683,0.16879
num__fico_range_low,-0.40267,0.00956,-42.11094,0.00000,0.66853,-0.42141,-0.38393
num__open_acc,0.01212,0.00720,1.68345,0.09229,1.01220,-0.00199,0.02623
num__pub_rec,-0.00289,0.00778,-0.37088,0.71073,0.99712,-0.01814,0.01236
num__revol_util,-0.01635,0.00678,-2.41100,0.01591,0.98378,-0.02965,-0.00306
num__mort_acc,-0.08407,0.00660,-12.73260,0.00000,0.91937,-0.09701,-0.07113
num__loan_amnt,0.09975,0.00823,12.11823,0.00000,1.10490,0.08362,0.11589
num__UNRATE,-0.03486,0.01195,-2.91639,0.00354,0.96574,-0.05828,-0.01143


## Phase 6: Evaluation Metrics, Probability Calibration & Statistical Uncertainty

### Evaluation Methodology on the Untouched 2015 Out-of-Time Test Set
1. **Class-Imbalanced Evaluation Metrics:**
   - **PR-AUC (Precision-Recall Area Under Curve):** Primary performance metric, benchmarked directly against the empirical default base rate ($P = 14.89\%$).
   - **ROC-AUC:** Discrimination metric evaluating borrower risk ranking across thresholds.
   - **Brier Score Loss ($BS = \frac{1}{N} \sum (\hat{p}_i - y_i)^2$):** Measures mean squared error of predicted default probabilities.
   - **Reliability Diagrams (`calibration_curve`):** Assesses probability calibration across predicted risk deciles.
   - **Classification Metrics:** Precision, Recall, F1-Score, and Confusion Matrix at default-prevalence threshold.
2. **Statistical Uncertainty & Shared Macro Observations:**
   - **Month-Clustered Standard Errors:** In `statsmodels.Logit`, standard errors are clustered by origination month (`issue_date`) to correct for the fact that millions of individual loans share 108 monthly macro observations.
   - **Time-Block Bootstrap Confidence Intervals:** Uncertainty in test metric deltas ($\Delta = \text{Metric}_{\text{Model C}} - \text{Metric}_{\text{Model B}}$) is estimated via **month-level block bootstrap resampling**, avoiding ordinary loan-level bootstrap which underestimates uncertainty.
   - **Small-Cluster Epistemic Limitation (G = 12):** The 2015 holdout test window contains exactly 12 calendar months. In small-cluster regimes (G < 30-50), cluster bootstrap confidence intervals and clustered standard errors have limited statistical power and downward variance bias. We explicitly document this constraint, avoid overstating statistical certainty, and corroborate holdout results with multi-year expanding-window CV (2013-2014).
3. **Predictive Association vs. Causality Disclaimer:**
   - All findings measure observational predictive uplift across credit vintages. Macro coefficients do NOT imply that macroeconomic policy shocks have direct causal effects on individual default behavior.

In [26]:
# Phase 6: Evaluation Metrics & Time-Block Bootstrap Uncertainty Calculation

from sklearn.metrics import (
    roc_auc_score, 
    precision_recall_curve, 
    auc, 
    brier_score_loss, 
    classification_report, 
    confusion_matrix
)
from sklearn.calibration import calibration_curve

print('Phase 6 evaluation and statistical uncertainty suite ready.')


Phase 6 evaluation and statistical uncertainty suite ready.


## Phase 7: ES 335 Deliverable — Macroeconomic Sensitivity Explorer & Web Export

### Client-Side Architecture & Sensitivity Explorer Framing
1. **Sensitivity Explorer, Not a Lifetime Forecasting Tool:**
   - The web tool evaluates how baseline origination default probability varies with macroeconomic conditions at issue, holding applicant characteristics constant.
   - Clearly discloses that it does **NOT** forecast default trajectories if macroeconomic conditions shift during repayment (evaluating lifetime economic shocks requires a survival/hazard model with time-varying covariates).
2. **Clamped Historical Training Bounds:**
   - Sliders in `web/index.html` are strictly bounded to observed 2007–2014 training values to prevent ungrounded extrapolation:
     * `UNRATE`: **4.0% to 10.0%** (2009 recession peak)
     * `CPI_YoY_Inflation`: **-2.0% to 5.5%**
     * `UNRATE_change_6m`: **-1.5% to +2.5%**
3. **Exact Preprocessing & Weight Export:**
   - Model C logistic regression coefficients ($\beta$) and intercept ($\beta_0$), alongside training feature means ($\mu_j$) and standard deviations ($\sigma_j$), are exported to `web/model_coefficients.json`.
   - Client-side JavaScript evaluates the standardized logistic sigmoid:
     $$P(\text{Default}) = \frac{1}{1 + e^{-(\beta_0 + \sum \beta_i \cdot \frac{X_i - \mu_i}{\sigma_i})}}$$
4. **Verification Requirement:**
   - Python predictions will be tested against JavaScript in-browser predictions to verify exact parity within $10^{-5}$ numerical tolerance.
   - LightGBM evaluation results are presented in static comparison tables; no fake linear approximations are used.

In [27]:
# Phase 7: Export Model Parameters to web/model_coefficients.json
# Exports exact fitted scaler means, standard deviations, and Logistic Regression weights.

import json
import os

print('Phase 7 export pipeline ready.')


Phase 7 export pipeline ready.
